# Notebook 01 - Tri des séances pertinentes

## Objectif
Parmi les **milliers de comptes rendus** de séances de l'Assemblée Nationale (au format XML), on ne retient que celles qui abordent les **violences sexistes et sexuelles (VSS)**.

## Méthode
Deux critères complémentaires permettent de sélectionner une séance :

1. **Tri par le sommaire** (`tri_titres`) : on cherche si un titre de point à l'ordre du jour contient un mot-clé VSS (ex : *"violences conjugales"*, *"harcèlement sexuel"*...).
2. **Tri par le texte** (`tri_textes`) : on compte le nombre d'occurrences de mots-clés VSS dans le corps du texte. Si au moins 3 mots-clés différents apparaissent, la séance est retenue.

Les séances sélectionnées sont copiées dans des dossiers `sorted/` séparés par législature.

## Entrées
- Fichiers XML bruts dans `data/CompteRendusXV/`, `data/CompteRendusXVI/`, `data/CompteRendusXVII/`

## Sorties
- Fichiers XML copiés dans `sorted/xv/`, `sorted/xvi/`, `sorted/xvii/`
- Liste des fichiers pertinents affichée en fin de notebook

## 1. Imports et configuration

In [19]:
from config import *
from lxml import etree
import os, re
from tqdm import tqdm
from shutil import copyfile

# Namespace XML de l'Assemblée Nationale
NAMESPACES = {'ns': 'http://schemas.assemblee-nationale.fr/referentiel'}

print("Configuration chargée.")

Configuration chargée.


## 2. Fonctions de tri

### 2.1 Tri par le sommaire
On parse le sommaire XML de chaque séance pour y chercher les mots-clés VSS dans les titres de niveau 1 et 2.

In [20]:
def tri_titres(fichier, liste):
    """
    Teste si le sommaire d'un fichier XML contient un des mots-clés de la liste.

    Args:
        fichier (str): chemin complet vers le fichier XML
        liste (list): liste de racines de mots à chercher

    Returns:
        bool: True si au moins un titre du sommaire contient un mot-clé
    """
    try:
        tree = etree.parse(fichier)
        root = tree.getroot()
        for chaine in liste:
            chaine_lower = chaine.lower()
            # Titres de niveau 1
            for titre in root.xpath('//ns:sommaire1//ns:intitule', namespaces=NAMESPACES):
                if titre.text and chaine_lower in titre.text.lower():
                    return True
            # Titres de niveau 2
            for titre in root.xpath('//ns:sommaire2//ns:intitule', namespaces=NAMESPACES):
                if titre.text and chaine_lower in titre.text.lower():
                    return True
        return False
    except Exception:
        return False

### 2.2 Tri par le texte
On parcourt tous les paragraphes du texte et on compte combien de mots-clés différents apparaissent. Si le seuil est atteint, la séance est retenue.

In [21]:
def tri_textes(fichier, liste, seuil):
    """
    Compte le nombre d'occurrences de mots-clés VSS dans le texte d'un fichier XML.

    Args:
        fichier (str): chemin complet vers le fichier XML
        liste (list): liste de racines de mots à chercher
        seuil (int): nombre minimum d'occurrences pour retenir le fichier

    Returns:
        bool: True si le nombre d'occurrences atteint le seuil
    """
    try:
        tree = etree.parse(fichier)
        root = tree.getroot()
        nb_occurences = 0
        for para in root.xpath('//ns:paragraphe', namespaces=NAMESPACES):
            text_nodes = para.xpath('.//ns:texte//text()', namespaces=NAMESPACES)
            texte_complet = " ".join(text_nodes).lower()
            for chaine in liste:
                pattern = rf'\b{re.escape(chaine.lower())}'
                nb_occurences += len(re.findall(pattern, texte_complet))
        return nb_occurences >= seuil
    except Exception:
        return False

## 3. Exécution du tri

On parcourt les 3 législatures. Pour chaque fichier XML, on applique les deux filtres. Si l'un des deux est positif, le fichier est ajouté à la liste des séances pertinentes.


In [22]:
fichier_cache = os.path.join(DOSSIER_DATAFRAMES, "liste_seances_pertinentes.txt")

if os.path.exists(fichier_cache):
    with open(fichier_cache, "r") as f:
        liste_pertinente = [line.strip() for line in f if line.strip()]
    print(f"Liste chargée depuis le cache : {len(liste_pertinente)} séances pertinentes.")
else:
    liste_pertinente = []

    for nom_leg, chemin_dossier in DOSSIERS_LEGISLATURES.items():
        if not os.path.exists(chemin_dossier):
            print(f"Dossier introuvable : {chemin_dossier}")
            continue

        fichiers_xml = [f for f in os.listdir(chemin_dossier) if f.endswith('.xml')]
        print(f"\nLégislature {nom_leg} : {len(fichiers_xml)} fichiers à analyser")

        for fichier in tqdm(fichiers_xml, desc=f"Tri {nom_leg}"):
            chemin_complet = os.path.join(chemin_dossier, fichier)

            if tri_titres(chemin_complet, A_TESTER):
                liste_pertinente.append(fichier)
                continue  # Pas besoin de tester le texte si le titre suffit

            if tri_textes(chemin_complet, A_TESTER, SEUIL_TRI):
                liste_pertinente.append(fichier)

    # Suppression des doublons (un même fichier pourrait être dans plusieurs dossiers)
    liste_pertinente = list(set(liste_pertinente))

    # Sauvegarde du cache
    with open(fichier_cache, "w") as f:
        f.write("\n".join(liste_pertinente))

    print(f"\nTri terminé : {len(liste_pertinente)} séances retenues.")
    print(f"Cache sauvegardé dans {fichier_cache}")

Liste chargée depuis le cache : 1524 séances pertinentes.


## 4. Copie des séances sélectionnées

On copie les fichiers retenus dans les dossiers `sorted/` pour les séparer des données brutes. Cela permet de ne travailler que sur les fichiers pertinents dans les notebooks suivants.

In [23]:
# ==========================================================================
# COPIE DES FICHIERS
# ==========================================================================

for nom_leg, chemin_source in DOSSIERS_LEGISLATURES.items():
    cle_sorted = nom_leg.lower()
    if cle_sorted not in DOSSIER_SORTED:
        continue

    chemin_dest = DOSSIER_SORTED[cle_sorted]
    os.makedirs(chemin_dest, exist_ok=True)

    if not os.path.exists(chemin_source):
        continue

    nb_copies = 0
    for fichier in os.listdir(chemin_source):
        if fichier in liste_pertinente:
            dest = os.path.join(chemin_dest, fichier)
            if not os.path.exists(dest):
                copyfile(os.path.join(chemin_source, fichier), dest)
                nb_copies += 1

    print(f"{nom_leg} : {nb_copies} nouveaux fichiers copiés dans {chemin_dest}")

print("\nCopie terminée.")

XV : 0 nouveaux fichiers copiés dans /home/onyxia/work/projet_eco_socio/sorted/xv
XVI : 0 nouveaux fichiers copiés dans /home/onyxia/work/projet_eco_socio/sorted/xvi
XVII : 0 nouveaux fichiers copiés dans /home/onyxia/work/projet_eco_socio/sorted/xvii

Copie terminée.


## 5. Vérification

In [24]:
# Vérification du contenu des dossiers sorted/
for leg, chemin in DOSSIER_SORTED.items():
    if os.path.exists(chemin):
        nb = len([f for f in os.listdir(chemin) if f.endswith('.xml')])
        print(f"  {leg.upper()} : {nb} fichiers XML")
    else:
        print(f"  {leg.upper()} : dossier non trouvé")

print(f"\nTotal de séances retenues : {len(liste_pertinente)}")

  XV : 1185 fichiers XML
  XVI : 458 fichiers XML
  XVII : 272 fichiers XML

Total de séances retenues : 1524
